# 01 — In Silico Digestion Exploration

This notebook tests the first end-to-end step of the HERALD computational pipeline: retrieving whey protein sequences and simulating enzymatic digestion.

The goal is to confirm that the current code can:

1. Retrieve target whey protein sequences from UniProt.
2. Apply enzyme-specific cleavage rules.
3. Generate predicted peptide fragments from each protein sequence.
4. Compare peptide profiles produced by different enzymes.

At this stage, the workflow does **not** yet predict antimicrobial activity. It only generates the peptide fragments that will later be scored for AMP-like properties.

Import HERALD Functions

This section imports the core functions developed so far.

* protein_sequence() retrieves a protein sequence from UniProt or the local cache.
* WHEY_PROTEINS stores the target whey protein names and their UniProt accession IDs.
* digest_sequence() simulates enzymatic digestion using the cleavage rules defined in enzymes.py.
* peptide_features() provides general peptide features.

In [3]:
import sys
import os

PROJECT_ROOT = os.path.abspath("..")
sys.path.append(PROJECT_ROOT)

from herald.proteins import protein_sequence, WHEY_PROTEINS
from herald.digestion import digest_sequence
from herald.scoring import peptide_features

Retrieve Whey Protein Sequences

Here, we retrieve the amino-acid sequences for the target whey proteins.

Each protein is identified by a UniProt accession ID. The sequence is either fetched directly from UniProt or loaded from the local cache if it has already been downloaded.

This step verifies that the protein-fetching module is working correctly and that the target proteins are available for downstream digestion analysis.

In [4]:
for protein_name, accession_id in WHEY_PROTEINS.items():
    sequence = protein_sequence(accession_id)
    print(protein_name, accession_id, len(sequence))

beta-lactoglobulin P02754 178
alpha-lactalbumin P00709 142
lactoferrin P24627 708
BSA P02769 607


## Simulate Trypsin Digestion of β-Lactoglobulin

In this section, we simulate digestion of β-lactoglobulin using trypsin.

Trypsin is a sequence-specific protease that primarily cleaves after lysine (`K`) and arginine (`R`), unless the next amino acid is proline (`P`). The resulting peptide fragments represent the predicted products of an in silico hydrolysis reaction.

For this initial test, only peptides between 4 and 50 amino acids are retained. This range removes very short fragments while keeping peptide-sized products that may later be evaluated for AMP-like properties.

In [5]:
beta_lg = protein_sequence(WHEY_PROTEINS["beta-lactoglobulin"])

trypsin_peptides = digest_sequence(
    beta_lg,
    "trypsin",
    min_length=4,
    max_length=50,
)

print(f"Number of trypsin peptides: {len(trypsin_peptides)}")
print("Trypsin peptides:", trypsin_peptides[:10])

Number of trypsin peptides: 13
Trypsin peptides: ['CLLLALALTCGAQALIVTQTMK', 'GLDIQK', 'VAGTWYSLAMAASDISLLDAQSAPLR', 'VYVEELKPTPEGDLEILLQK', 'WENGECAQK', 'IIAEK', 'IPAVFK', 'IDALNENK', 'VLVLDTDYK', 'YLLFCMENSAEPEQSLACQCLVR']


## Compare Trypsin and Chymotrypsin Digestion

Different enzymes cleave proteins at different amino-acid positions. As a result, the same starting protein can produce different peptide profiles depending on the enzyme used.

Here, we compare predicted peptide fragments from β-lactoglobulin digested with:

- **Trypsin**, which cleaves primarily after `K` and `R`
- **Chymotrypsin**, which cleaves primarily after aromatic or hydrophobic residues such as `F`, `Y`, `W`, and `L`

This comparison provides an initial check that the enzyme-specific cleavage rules produce different computational digestion outcomes.

In [6]:
chymotrypsin_peptides = digest_sequence(
    beta_lg,
    "chymotrypsin",
    min_length=4,
    max_length=50,
)

print(f"Number of trypsin peptides: {len(trypsin_peptides)}")
print(f"Number of chymotrypsin peptides: {len(chymotrypsin_peptides)}")
print("Chymotrypsin peptides:", chymotrypsin_peptides[:10])

Number of trypsin peptides: 13
Number of chymotrypsin peptides: 19
Chymotrypsin peptides: ['MKCL', 'TCGAQAL', 'IVTQTMKGL', 'DIQKVAGTW', 'AMAASDISL', 'DAQSAPL', 'VEEL', 'KPTPEGDL', 'ENGECAQKKIIAEKTKIPAVF', 'KIDAL']


## Compute Peptide Features

The digestion step produces a list of predicted peptide fragments, but it does not yet indicate which peptides may be biologically interesting.

In this section, we compute simple AMP-relevant features for each predicted peptide. These first-pass features include:

- **Length** — antimicrobial peptides are often short to medium-sized peptides.
- **Net charge** — many AMPs are positively charged, which can help them interact with negatively charged bacterial membranes.
- **Hydrophobic fraction** — hydrophobic residues can contribute to membrane interaction.
- **Aromatic fraction** — aromatic residues such as `F`, `W`, and `Y` may also contribute to peptide–membrane interactions.

These calculations are intentionally simple and transparent. They provide an initial screening layer before adding more advanced AMP prediction models. The residue groupings are simplified biochemical categories used for initial screening. Net charge is approximated from basic and acidic residue counts, while hydrophobic and aromatic fractions are computed from selected residue sets commonly associated with peptide–membrane interactions. These features are useful for transparent exploratory ranking, but later versions should compare against pH-dependent charge calculations, hydrophobicity scales, and trained AMP prediction models.

In [13]:
trypsin_features = [peptide_features(peptide) for peptide in trypsin_peptides]
trypsin_features[:5]

[{'sequence': 'CLLLALALTCGAQALIVTQTMK',
  'length': 22,
  'net_charge': 1,
  'hydrophobic_fraction': 0.5909090909090909,
  'aromatic_fraction': 0.0,
  'simple_amp_score': 3},
 {'sequence': 'GLDIQK',
  'length': 6,
  'net_charge': 0,
  'hydrophobic_fraction': 0.3333333333333333,
  'aromatic_fraction': 0.0,
  'simple_amp_score': 1},
 {'sequence': 'VAGTWYSLAMAASDISLLDAQSAPLR',
  'length': 26,
  'net_charge': -1,
  'hydrophobic_fraction': 0.5769230769230769,
  'aromatic_fraction': 0.07692307692307693,
  'simple_amp_score': 3},
 {'sequence': 'VYVEELKPTPEGDLEILLQK',
  'length': 20,
  'net_charge': -3,
  'hydrophobic_fraction': 0.4,
  'aromatic_fraction': 0.05,
  'simple_amp_score': 3},
 {'sequence': 'WENGECAQK',
  'length': 9,
  'net_charge': -1,
  'hydrophobic_fraction': 0.2222222222222222,
  'aromatic_fraction': 0.1111111111111111,
  'simple_amp_score': 2}]

## Rank Peptides by Simple AMP-Like Score

The peptide feature table can now be used to rank predicted hydrolysis products.

Here, peptides are sorted by the `simple_amp_score`, a transparent heuristic that rewards broad AMP-like properties such as moderate length, positive charge, moderate hydrophobicity, and aromatic content.

This score is not a validated AMP predictor. It is used as an exploratory baseline to identify candidate peptides for later comparison against AMP databases, machine-learning models, and experimental results.

In [14]:
ranked_trypsin_peptides = sorted(
    trypsin_features,
    key=lambda x: x["simple_amp_score"],
    reverse=True,
)

for peptide in ranked_trypsin_peptides[:10]:
    print(
        peptide["sequence"],
        "score:", peptide["simple_amp_score"],
        "charge:", peptide["net_charge"],
        "hydrophobic:", round(peptide["hydrophobic_fraction"], 2),
        "aromatic:", round(peptide["aromatic_fraction"], 2),
    )

CLLLALALTCGAQALIVTQTMK score: 3 charge: 1 hydrophobic: 0.59 aromatic: 0.0
VAGTWYSLAMAASDISLLDAQSAPLR score: 3 charge: -1 hydrophobic: 0.58 aromatic: 0.08
VYVEELKPTPEGDLEILLQK score: 3 charge: -3 hydrophobic: 0.4 aromatic: 0.05
IPAVFK score: 3 charge: 1 hydrophobic: 0.67 aromatic: 0.17
VLVLDTDYK score: 3 charge: -1 hydrophobic: 0.56 aromatic: 0.11
YLLFCMENSAEPEQSLACQCLVR score: 3 charge: -2 hydrophobic: 0.43 aromatic: 0.09
ALPMHIR score: 3 charge: 2 hydrophobic: 0.57 aromatic: 0.0
WENGECAQK score: 2 charge: -1 hydrophobic: 0.22 aromatic: 0.11
IDALNENK score: 2 charge: -1 hydrophobic: 0.38 aromatic: 0.0
LSFNPTQLEEQCHI score: 2 charge: -1 hydrophobic: 0.29 aromatic: 0.07


In [15]:
chymotrypsin_features = [peptide_features(peptide) for peptide in chymotrypsin_peptides]

ranked_chymotrypsin_peptides = sorted(
    chymotrypsin_features,
    key=lambda x: x["simple_amp_score"],
    reverse=True,
)

for peptide in ranked_chymotrypsin_peptides[:10]:
    print(
        peptide["sequence"],
        "score:", peptide["simple_amp_score"],
        "charge:", peptide["net_charge"],
        "hydrophobic:", round(peptide["hydrophobic_fraction"], 2),
        "aromatic:", round(peptide["aromatic_fraction"], 2),
    )

ENGECAQKKIIAEKTKIPAVF score: 4 charge: 1 hydrophobic: 0.38 aromatic: 0.05
KALPMHIRL score: 4 charge: 3 hydrophobic: 0.56 aromatic: 0.0
IVTQTMKGL score: 3 charge: 1 hydrophobic: 0.44 aromatic: 0.0
DIQKVAGTW score: 3 charge: 0 hydrophobic: 0.44 aromatic: 0.11
MKCL score: 2 charge: 1 hydrophobic: 0.5 aromatic: 0.0
AMAASDISL score: 2 charge: -1 hydrophobic: 0.67 aromatic: 0.0
VRTPEVDDEAL score: 2 charge: -3 hydrophobic: 0.36 aromatic: 0.0
TCGAQAL score: 1 charge: 0 hydrophobic: 0.43 aromatic: 0.0
DAQSAPL score: 1 charge: -1 hydrophobic: 0.43 aromatic: 0.0
VEEL score: 1 charge: -2 hydrophobic: 0.5 aromatic: 0.0


## Summarize Enzyme-Level Outputs

After ranking individual peptides, we can begin comparing enzymes at the level of their overall peptide profiles.

This section summarizes the predicted digestion output for each enzyme by reporting:

- the total number of retained peptide fragments
- the highest simple AMP-like score among the generated peptides

This provides a first enzyme-level comparison. At this stage, the comparison is still exploratory because the `simple_amp_score` is a rule-based baseline rather than a validated predictor of antimicrobial activity.

In [16]:
print("Trypsin")
print("Number of peptides:", len(trypsin_peptides))
print("Top score:", ranked_trypsin_peptides[0]["simple_amp_score"])

print("\nChymotrypsin")
print("Number of peptides:", len(chymotrypsin_peptides))
print("Top score:", ranked_chymotrypsin_peptides[0]["simple_amp_score"])

Trypsin
Number of peptides: 13
Top score: 3

Chymotrypsin
Number of peptides: 19
Top score: 4
